# Chapter 2 Practical 07: Context, Explainability, and Zero-Shot Demo

Learning objectives:
- Apply context-aware re-ranking.
- Explain recommendations with shared content features.
- Build a zero-shot style text search interface.
- Keep generative enrichment as an optional stub, not a required API call.

Slide connection: context-aware recommendation, explainable recommendation, zero-shot examples, and generative metadata enrichment.


Load the same movie data so this notebook connects back to the earlier practicals.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Create a base TF-IDF recommender. This acts as the reliable fallback for zero-shot text queries.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["search_text"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
)

vectorizer = TfidfVectorizer(stop_words="english")
item_matrix = vectorizer.fit_transform(movies["search_text"])


Zero-shot style search lets the user describe what they want instead of choosing a seed item.


In [ ]:
def search_movies(query, n=6):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, item_matrix).ravel()
    results = movies[["title", "genres", "director", "family_friendly", "duration_min"]].copy()
    results["base_score"] = scores
    return results.sort_values("base_score", ascending=False).head(n)

search_movies("movies about space exploration")


Context-aware recommendation adjusts the ranking for the current situation.


In [ ]:
def rerank_for_context(results, context):
    adjusted = results.copy()
    adjusted["context_bonus"] = 0.0

    if context == "morning_mobile":
        adjusted.loc[adjusted["duration_min"] <= 110, "context_bonus"] += 0.12
    elif context == "evening_tv":
        adjusted.loc[adjusted["duration_min"] >= 120, "context_bonus"] += 0.10
    elif context == "family_mode":
        adjusted.loc[adjusted["family_friendly"] == 1, "context_bonus"] += 0.20

    adjusted["final_score"] = adjusted["base_score"] + adjusted["context_bonus"]
    return adjusted.sort_values("final_score", ascending=False)

base = search_movies("light comedy for family evening", n=8)
rerank_for_context(base, "family_mode")


Explanations should be short and specific. Here we explain by shared genres, director, and high-weight query terms.


In [ ]:
def explain_with_features(query, title, top_terms=5):
    movie = movies[movies["title"].eq(title)].iloc[0]
    query_vector = vectorizer.transform([query]).toarray().ravel()
    movie_vector = item_matrix[movies.index[movies["title"].eq(title)][0]].toarray().ravel()
    contribution = query_vector * movie_vector
    terms = vectorizer.get_feature_names_out()
    best_terms = [terms[i] for i in contribution.argsort()[::-1][:top_terms] if contribution[i] > 0]

    return {
        "movie": title,
        "recommended_because_it_shares": ", ".join(best_terms) if best_terms else "related content features",
        "genres": movie["genres"],
        "director": movie["director"],
    }

query = "space survival astronaut"
top_title = search_movies(query, n=1).iloc[0]["title"]
explain_with_features(query, top_title)


A contribution table makes the explanation inspectable rather than magical.


In [ ]:
def contribution_table(query, title):
    idx = movies.index[movies["title"].eq(title)][0]
    q = vectorizer.transform([query]).toarray().ravel()
    x = item_matrix[idx].toarray().ravel()
    terms = vectorizer.get_feature_names_out()
    table = pd.DataFrame({"term": terms, "query_weight": q, "movie_weight": x})
    table["contribution"] = table["query_weight"] * table["movie_weight"]
    return table[table["contribution"] > 0].sort_values("contribution", ascending=False).head(10)

contribution_table("space survival astronaut", top_title)


Optional SBERT can replace TF-IDF for zero-shot search if it is available.


In [ ]:
semantic_search_available = False
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(movies["search_text"].tolist(), show_progress_bar=False)
    semantic_search_available = True
except Exception as exc:
    print("Semantic search model is optional and not available here. TF-IDF search remains active.")
    print(type(exc).__name__, str(exc)[:160])

semantic_search_available


Generative enrichment can be introduced as a stub. Students should not need an API key to run the notebook.


In [ ]:
def enrich_metadata_stub(title):
    return {
        "title": title,
        "possible_extra_tags": ["teaching stub", "replace with reviewed metadata", "no API call required"],
        "note": "In production, generated metadata should be checked before it affects recommendations.",
    }

enrich_metadata_stub("Interstellar")


## What did we learn?

- Context can re-rank otherwise reasonable recommendations.
- Explanations should name concrete shared features.
- Zero-shot search can be taught with TF-IDF first and upgraded to embeddings when available.
- Generative enrichment is powerful, but it should be optional and reviewed.

Exercises:
1. Add a `late_night` context and define your own re-ranking rule.
2. Write two natural-language queries and compare their recommendation lists.
